[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/semantic-router/blob/main/docs/06-calibration.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/aurelio-labs/semantic-router/blob/main/docs/06-calibration.ipynb)

# Route Calibration

A router's `confidence` decides whether a route is chosen: if it clears the `confidence_threshold`, the route passes, otherwise we get _no_ route.

Out of the box, `IndexedRouter` reports the raw similarity between a query and a route's utterances. A raw `0.7` means something different for every encoder, set of routes and dataset, so a threshold on raw scores can only be found by trial and error.

`fit()` fixes that. Given a few labelled queries, it learns a map from raw score to the probability that the top route is right. After fitting, a confidence of `0.8` means the router is right about 80% of the time at that score, so we can set the threshold from how costly a wrong route is. The map only ever rises, so it never changes which route ranks first. We measure the result with `evaluate()`.

In [ ]:
!pip install -qU "semantic-router[fastembed]>=1.0.0"

## Define the Router

We start by defining four routes; _politics_, _chitchat_, _mathematics_, and _biology_.

In [1]:
from semantic_router import Route

# we could use this as a guide for our chatbot to avoid political conversations
politics = Route(
    name="politics",
    utterances=[
        "isn't politics the best thing ever",
        "why don't you tell me about your political opinions",
        "don't you just love the president",
        "don't you just hate the president",
        "they're going to destroy this country!",
        "they will save the country!",
    ],
)

# this could be used as an indicator to our chatbot to switch to a more
# conversational prompt
chitchat = Route(
    name="chitchat",
    utterances=[
        "Did you watch the game last night?",
        "what's your favorite type of music?",
        "Have you read any good books lately?",
        "nice weather we're having",
        "Do you have any plans for the weekend?",
    ],
)

# we can use this to switch to an agent with more math tools, prompting, and LLMs
mathematics = Route(
    name="mathematics",
    utterances=[
        "can you explain the concept of a derivative?",
        "What is the formula for the area of a triangle?",
        "how do you solve a system of linear equations?",
        "What is the concept of a prime number?",
        "Can you explain the Pythagorean theorem?",
    ],
)

# we can use this to switch to an agent with more biology knowledge
biology = Route(
    name="biology",
    utterances=[
        "what is the process of osmosis?",
        "can you explain the structure of a cell?",
        "What is the role of RNA?",
        "What is genetic mutation?",
        "Can you explain the process of photosynthesis?",
    ],
)

# we place all of our decisions together into single list
routes = [politics, chitchat, mathematics, biology]

For our encoder we will use the local `FastEmbedEncoder`, then build the router.

In [2]:
from semantic_router import IndexedRouter
from semantic_router.encoders import FastEmbedEncoder

encoder = FastEmbedEncoder(name="BAAI/bge-small-en-v1.5")

router = IndexedRouter(encoder=encoder, routes=routes)

Before fitting, every confidence is a raw similarity:

In [3]:
queries = [
    "don't you love politics?",
    "What's DNA?",
    "I'm interested in learning about llama 2",
    "what is the capital of Spain?",
]


def show(router):
    for query in queries:
        choice = router(query)
        print(
            f"{query} -> {choice.name} ({choice.confidence:.2f}, calibrated={choice.calibrated})"
        )


show(router)

don't you love politics? -> politics (0.70, calibrated=False)
What's DNA? -> biology (0.65, calibrated=False)
I'm interested in learning about llama 2 -> biology (0.52, calibrated=False)
what is the capital of Spain? -> politics (0.42, calibrated=False)


The last two queries don't belong to any route, but with no threshold the router still returns one. Their scores are lower than the real matches, but by how much, and where should the bar go?

## Labelled Queries

To calibrate, we need queries labelled with the route they should go to. A label of `None` means no route should match. These queries must be **held out**: none of them are utterances stored in the router (we will see why below).

_Hint: try using an LLM to generate examples for your own use-case, or better, label some real traffic._

In [4]:
data = [
    # politics
    ("What's your opinion on the current government?", "politics"),
    ("Who do you think will win the next election?", "politics"),
    ("What are your thoughts on the new policy?", "politics"),
    ("How do you feel about the political situation?", "politics"),
    ("Do you agree with the president's actions?", "politics"),
    ("What's your stance on the political debate?", "politics"),
    ("How do you see the future of our country?", "politics"),
    ("What do you think about the opposition party?", "politics"),
    ("Do you believe the government is doing enough?", "politics"),
    ("What's your opinion on the political scandal?", "politics"),
    ("Do you think the new law will make a difference?", "politics"),
    ("What are your thoughts on the political reform?", "politics"),
    ("Do you agree with the government's foreign policy?", "politics"),
    # chitchat
    ("What's the weather like?", "chitchat"),
    ("It's a beautiful day today.", "chitchat"),
    ("How's your day going?", "chitchat"),
    ("It's raining cats and dogs.", "chitchat"),
    ("Let's grab a coffee.", "chitchat"),
    ("What's up?", "chitchat"),
    ("It's a bit chilly today.", "chitchat"),
    ("How's it going?", "chitchat"),
    ("Nice weather we're having.", "chitchat"),
    ("It's a bit windy today.", "chitchat"),
    ("Let's go for a walk.", "chitchat"),
    ("How's your week been?", "chitchat"),
    ("It's quite sunny today.", "chitchat"),
    ("How are you feeling?", "chitchat"),
    ("It's a bit cloudy today.", "chitchat"),
    # mathematics
    ("What is the Pythagorean theorem?", "mathematics"),
    ("Can you solve this quadratic equation?", "mathematics"),
    ("What is the derivative of x squared?", "mathematics"),
    ("Explain the concept of integration.", "mathematics"),
    ("What is the area of a circle?", "mathematics"),
    ("How do you calculate the volume of a sphere?", "mathematics"),
    ("What is the difference between a vector and a scalar?", "mathematics"),
    ("Explain the concept of a matrix.", "mathematics"),
    ("What is the Fibonacci sequence?", "mathematics"),
    ("How do you calculate permutations?", "mathematics"),
    ("What is the concept of probability?", "mathematics"),
    ("Explain the binomial theorem.", "mathematics"),
    ("What is the difference between discrete and continuous data?", "mathematics"),
    ("What is a complex number?", "mathematics"),
    ("Explain the concept of limits.", "mathematics"),
    # biology
    ("What is photosynthesis?", "biology"),
    ("Explain the process of cell division.", "biology"),
    ("What is the function of mitochondria?", "biology"),
    ("What is DNA?", "biology"),
    ("What is the difference between prokaryotic and eukaryotic cells?", "biology"),
    ("What is an ecosystem?", "biology"),
    ("Explain the theory of evolution.", "biology"),
    ("What is a species?", "biology"),
    ("What is the role of enzymes?", "biology"),
    ("What is the circulatory system?", "biology"),
    ("Explain the process of respiration.", "biology"),
    ("What is a gene?", "biology"),
    ("What is the function of the nervous system?", "biology"),
    ("What is homeostasis?", "biology"),
    ("What is the difference between a virus and a bacteria?", "biology"),
    ("What is the role of the immune system?", "biology"),
    # queries that no route should match
    ("What is the capital of France?", None),
    ("how many people live in the US?", None),
    ("when is the best time to visit Bali?", None),
    ("how do I learn a language", None),
    ("tell me an interesting fact", None),
    ("what is the best programming language?", None),
    ("I'm interested in learning about llama 2", None),
    ("can you recommend a good pizza place nearby?", None),
    ("how do I reset my router password?", None),
    ("what's the plot of the latest Marvel movie?", None),
    ("translate 'good morning' into Spanish", None),
    ("how long should I boil an egg?", None),
    ("what's a good name for a golden retriever?", None),
    ("how do I fix a flat bike tyre?", None),
    ("write me a haiku about autumn", None),
    ("which laptop should I buy for video editing?", None),
    ("how do I open a bank account?", None),
    ("what time does the train to Manchester leave?", None),
    ("explain how a car engine works", None),
    ("what are the rules of cricket?", None),
]

We split them in two: one half to fit the calibrator, the other to test it.

In [5]:
fit_data, test_data = data[::2], data[1::2]

X_fit, y_fit = zip(*fit_data)
X_test, y_test = zip(*test_data)

len(fit_data), len(test_data)

(40, 39)

`evaluate` gives the router's accuracy on labelled queries, with its threshold applied. With no threshold the router always returns a route, so every `None` query is a miss:

In [6]:
accuracy = router.evaluate(X=X_test, y=y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 66.67%


## Calibrate with `fit`

Now we call `fit` with our held-out queries and their labels:

In [7]:
report = router.fit(X=X_fit, y=y_fit, method="platt")
report

FitReport(method='platt', n=40, accuracy=1.0, ece=0.077977021916979)

The `FitReport` tells us:

- `n`: how many labelled queries the calibrator was fitted on.
- `accuracy`: how often the top route was right, for queries that have a route (before any threshold).
- `ece`: the expected calibration error, the average gap between the confidence the router states and how often it is actually right. Lower is better.

The same queries now come back with calibrated confidences:

In [8]:
show(router)

don't you love politics? -> politics (1.00, calibrated=True)
What's DNA? -> biology (0.99, calibrated=True)
I'm interested in learning about llama 2 -> biology (0.36, calibrated=True)
what is the capital of Spain? -> politics (0.01, calibrated=True)


The real matches are now near-certain, and the two unrelated queries are clearly unsure. So we can pick a threshold that means something: `0.5` returns a route only when it is more likely to be right than wrong.

In [9]:
router.confidence_threshold = 0.5

accuracy = router.evaluate(X=X_test, y=y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 84.62%


That is _much_ better, on queries the calibrator never saw. If a wrong route is costly, raise the bar. Some routes need more certainty than others, so a `Route` can carry its own `confidence_threshold`, which beats the router's:

```python
politics = Route(
    name="politics",
    utterances=[...],
    confidence_threshold=0.9,
)
```

## Why Held-Out Queries?

It is tempting to calibrate on the utterances we already have. Let's see what happens if we do, using the stored utterances plus the same `None` queries:

In [10]:
stored = [(utterance, route.name) for route in routes for utterance in route.utterances]
no_route = [(query, label) for query, label in fit_data if label is None]
X_stored, y_stored = zip(*(stored + no_route))

self_fit = IndexedRouter(encoder=encoder, routes=routes)
self_fit.fit(X=X_stored, y=y_stored)

2026-10-15 15:03:36 WARNING semantic_router 21/31 training queries are stored utterances; they match themselves at ~1.0, so the calibrator will under-rate real queries, which score lower. fit on held-out queries.


FitReport(method='platt', n=31, accuracy=1.0, ece=0.08919822058904267)

The router warns us. A stored utterance matches itself, so it scores higher than any real query will. The calibrator learns that only those scores are right, so real queries, which score lower, come out unsure:

In [11]:
show(self_fit)

self_fit.confidence_threshold = 0.5
accuracy = self_fit.evaluate(X=X_test, y=y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

don't you love politics? -> politics (0.95, calibrated=True)
What's DNA? -> biology (0.86, calibrated=True)
I'm interested in learning about llama 2 -> biology (0.19, calibrated=True)
what is the capital of Spain? -> politics (0.02, calibrated=True)


Accuracy: 71.79%


Its own `ece` still looks fine, which is what makes this easy to miss. Always fit on queries the router hasn't stored, and test on a separate set.

## Platt or Isotonic

`fit` supports two methods. `method="platt"` (the default) fits an S-shaped curve with two numbers, so it works with tens of labelled queries. `method="isotonic"` fits a free-form step curve: use it when you have hundreds of labelled queries, otherwise it overfits into flat steps.

When you `add` or `remove` routes, the router refits its calibrator on the same labelled queries, and warns you if a route has none. The calibrator is also saved with the router, see [saving and loading routers](https://github.com/aurelio-labs/semantic-router/blob/main/docs/01-save-load-from-file.ipynb).

---